In [20]:
import numpy as np
import pandas as pd
import models
from database import engine, sessionDB
from nba_api.stats.static import players
from nba_api.stats.endpoints import playerawards

In [21]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [22]:
players= players.get_players()
#players

In [23]:
models.Base.metadata.create_all(bind=engine)
db = sessionDB()

award testing and structure allat stuff

In [24]:
#using lebron cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544
lebron_awards = playerawards.PlayerAwards(player_id=lebron_id).get_data_frames()[0]

acronym_hmap = {
    "All-Defensive Team": "DEF",''
    "All-NBA": "NBA",
    "All-Rookie Team": "ROOK",
    "NBA All-Star": "AS",
    "NBA All-Star Most Valuable Player": "ASMVP",
    "NBA Champion": "CHAMP",
    "NBA Cup All-Tournament Team": "CUP",
    "NBA Cup Most Valuable Player": "CUPMVP",
    "NBA Finals Most Valuable Player": "FMVP",
    "NBA Most Valuable Player": "MVP",
    "NBA Player of the Month": "POTM",
    "NBA Player of the Week": "POTW",
    "NBA Rookie of the Month": "ROTM",
    "NBA Rookie of the Year": "ROTY",
}

all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

#gotta add stat champs + dpoy/dpom + cfmvp + cpoy + dunk/3pt contest + teammate oty + hustle player oty

clean_lebron_awards_df = lebron_awards[lebron_awards["DESCRIPTION"].isin(acronym_hmap)]
clean_lebron_awards_df = clean_lebron_awards_df.drop([
    "FIRST_NAME", 
    "LAST_NAME", 
    "CONFERENCE", 
    "TYPE", 
    "SUBTYPE1", 
    "SUBTYPE2", 
    "SUBTYPE3",
    "MONTH",
    "WEEK",
    "PERSON_ID",
    "TEAM"
], axis=1)

clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"] = clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
clean_lebron_awards_df = clean_lebron_awards_df.replace(r'^\s*$', 0, regex=True) 
#^ gets rid of empty strings

clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df["DESCRIPTION"].map(acronym_hmap)
clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df.apply(
    lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
    if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
    axis=1
)

#clean_lebron_awards_df

actual award parsing

In [28]:
def getAwards(pid: int):
    awards_df = playerawards.PlayerAwards(player_id=pid).get_data_frames()[0]

    acronym_hmap = {
        "All-Defensive Team": "DEF",''
        "All-NBA": "NBA",
        "All-Rookie Team": "ROOK",
        "NBA All-Star": "AS",
        "NBA All-Star Most Valuable Player": "ASMVP",
        "NBA Champion": "CHAMP",
        "NBA Cup All-Tournament Team": "CUP",
        "NBA Cup Most Valuable Player": "CUPMVP",
        "NBA Finals Most Valuable Player": "FMVP",
        "NBA Most Valuable Player": "MVP",
        "NBA Player of the Month": "POTM",
        "NBA Player of the Week": "POTW",
        "NBA Rookie of the Month": "ROTM",
        "NBA Rookie of the Year": "ROTY",
    }

    all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

    awards_df = awards_df[awards_df["DESCRIPTION"].isin(acronym_hmap)]
    awards_df = awards_df.drop([
        "FIRST_NAME", 
        "LAST_NAME", 
        "CONFERENCE", 
        "TYPE", 
        "SUBTYPE1", 
        "SUBTYPE2", 
        "SUBTYPE3",
        "MONTH",
        "WEEK",
        "PERSON_ID",
        "TEAM"
    ], axis=1)

    awards_df["ALL_NBA_TEAM_NUMBER"] = awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
    awards_df = awards_df.replace(r'^\s*$', 0, regex=True) 
    #^ gets rid of empty strings

    awards_df["DESCRIPTION"] = awards_df["DESCRIPTION"].map(acronym_hmap)
    awards_df["DESCRIPTION"] = awards_df.apply(
        lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
        if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
        axis=1
    )

    return awards_df.drop(["ALL_NBA_TEAM_NUMBER"], axis=1)

bron_id = 2544

awards = getAwards(bron_id)

awards = [
    models.Award(
        player_id = bron_id, #dont hardcode this in prod
        season = award.SEASON,
        award_name = award.DESCRIPTION
    )
    for award in awards.itertuples()
]

db.add_all(awards)
db.commit()
db.close()

PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (psycopg2.errors.ForeignKeyViolation) insert or update on table "awards" violates foreign key constraint "awards_player_id_fkey"
DETAIL:  Key (player_id)=(2544) is not present in table "players".

[SQL: INSERT INTO awards (player_id, season, award_name) SELECT p0::INTEGER, p1::VARCHAR, p2::VARCHAR FROM (VALUES (%(player_id__0)s, %(season__0)s, %(award_name__0)s, 0), (%(player_id__1)s, %(season__1)s, %(award_name__1)s, 1), (%(player_id__2)s, %(season ... 11245 characters truncated ... 1)) AS imp_sen(p0, p1, p2, sen_counter) ORDER BY sen_counter RETURNING awards.id, awards.id AS id__1]
[parameters: {'season__0': '2008-09', 'award_name__0': 'DEF1', 'player_id__0': 2544, 'season__1': '2009-10', 'award_name__1': 'DEF1', 'player_id__1': 2544, 'season__2': '2010-11', 'award_name__2'